# IMPORT THE LIBRARYS FOR PROJECT

In [2]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer, make_column_selector
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler, PolynomialFeatures
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score,confusion_matrix, precision_score,f1_score


# LOAD THE DATASET

In [3]:
df=pd.read_csv("mEzNPN.csv")

In [4]:
df

,Loan_ID,Gender,Married,Dependents,Education,Self_Employed,ApplicantIncome,CoapplicantIncome,LoanAmount,Loan_Amount_Term,Credit_History,Property_Area,Loan_Status
0,LP001003,Male,Yes,1,Graduate,No,4583.0,1508.0,128.0,360.0,1.0,Rural,N
1,LP001005,Male,Yes,0,Graduate,Yes,3000.0,0.0,66.0,360.0,1.0,Urban,Y
2,LP001006,Male,Yes,0,Not Graduate,No,2583.0,2358.0,120.0,360.0,1.0,Urban,Y
3,LP001008,Male,No,0,Graduate,No,6000.0,0.0,141.0,360.0,1.0,Urban,Y
4,LP001013,Male,Yes,0,Not Graduate,No,2333.0,1516.0,95.0,360.0,1.0,Urban,Y
...,...,...,...,...,...,...,...,...,...,...,...,...,...
376,LP002953,Male,Yes,3+,Graduate,No,5703.0,0.0,128.0,360.0,1.0,Urban,Y
377,LP002974,Male,Yes,0,Graduate,No,3232.0,1950.0,108.0,360.0,1.0,Rural,Y
378,LP002978,Female,No,0,Graduate,No,2900.0,0.0,71.0,360.0,1.0,Rural,Y
379,LP002979,Male,Yes,3+,NaN,NaN,4106.0,0.0,40.0,180.0,1.0,Rural,Y


# MAPING THE CATOGORICAL COLUMNS

In [5]:
df.Gender=df.Gender.map({
    'Male':1,
    'Female':0
    
})

In [6]:
df.Loan_Status.isna().sum()

np.int64(0)

In [7]:
df.Education=df.Education.map({
    'Graduate':1,
    'Not Graduate':0
    
})

In [8]:
df.Self_Employed=df.Self_Employed.map({
    "Yes":1,
    "No":0
})

In [9]:
df.Property_Area=df.Property_Area.map({
    'Urban':1,
    "Rural":0
})

In [10]:
df.Married=df.Married.map({
    'Yes':1,
    'No':0
})

In [11]:
df.Loan_Status=df.Loan_Status.replace({
    'Y':1,
    'N':0
})

C:\Users\PMLS\AppData\Local\Temp\ipykernel_23684\324800776.py:1: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df.Loan_Status=df.Loan_Status.replace({


In [12]:
df.head()

,Loan_ID,Gender,Married,Dependents,Education,Self_Employed,ApplicantIncome,CoapplicantIncome,LoanAmount,Loan_Amount_Term,Credit_History,Property_Area,Loan_Status
0,LP001003,1.0,1,1,1.0,0.0,4583.0,1508.0,128.0,360.0,1.0,0.0,0
1,LP001005,1.0,1,0,1.0,1.0,3000.0,0.0,66.0,360.0,1.0,1.0,1
2,LP001006,1.0,1,0,0.0,0.0,2583.0,2358.0,120.0,360.0,1.0,1.0,1
3,LP001008,1.0,0,0,1.0,0.0,6000.0,0.0,141.0,360.0,1.0,1.0,1
4,LP001013,1.0,1,0,0.0,0.0,2333.0,1516.0,95.0,360.0,1.0,1.0,1


In [13]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 381 entries, 0 to 380
Data columns (total 13 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   Loan_ID            381 non-null    object 
 1   Gender             376 non-null    float64
 2   Married            381 non-null    int64  
 3   Dependents         373 non-null    object 
 4   Education          377 non-null    float64
 5   Self_Employed      358 non-null    float64
 6   ApplicantIncome    377 non-null    float64
 7   CoapplicantIncome  376 non-null    float64
 8   LoanAmount         379 non-null    float64
 9   Loan_Amount_Term   368 non-null    float64
 10  Credit_History     351 non-null    float64
 11  Property_Area      232 non-null    float64
 12  Loan_Status        381 non-null    int64  
dtypes: float64(9), int64(2), object(2)
memory usage: 38.8+ KB


In [14]:
df.duplicated().sum()

np.int64(0)

In [15]:
df.isna().sum()

Loan_ID                0
Gender                 5
Married                0
Dependents             8
Education              4
Self_Employed         23
ApplicantIncome        4
CoapplicantIncome      5
LoanAmount             2
Loan_Amount_Term      13
Credit_History        30
Property_Area        149
Loan_Status            0
dtype: int64

# DELEATE THE UNWANTED COLUMNS FROM DATASET

In [16]:
df=df.drop('Loan_ID',axis=1)

# SEPERATE THE DATA IN X (FEATURES) AND Y (LABELS)

In [17]:
X=df.drop('Loan_Status',axis=1)

In [18]:
y=df['Loan_Status']

# PIPELINE FOR LOAN APPOROVAL DATASET

In [19]:
number_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="median")), # yaha pa
    ("scaler", StandardScaler())
])

catagorical_pipeline = Pipeline([
    ("imputer",SimpleImputer(strategy="most_frequent")),
    ("encoder",OneHotEncoder(handle_unknown="ignore",sparse_output=False))
])

In [20]:
transformer = ColumnTransformer([
    ("num", number_pipeline, make_column_selector(dtype_include=np.number)),# ya column_selector uper pipeline ma data bajtaha number wala
    ("cat", catagorical_pipeline, make_column_selector(dtype_include=object))# ya catagorical pipeline ma bajta ha ya kam ha tranformer ka
])

In [21]:
model_pipeline = Pipeline([
                            ("preprocessing", transformer),
                            ("model", LogisticRegression())
])

# DATA SPLIT INTO TRAIN AND TEST 

In [22]:
X_train,X_test,y_train,y_test = train_test_split(X,y,test_size=0.2,random_state=36)

# FIT THE MODEL ON TRAINING DATA

In [23]:
model_pipeline.fit(X_train,y_train)

,steps,"[('preprocessing', ...), ('model', ...)]"
,transform_input,None
,memory,None
,verbose,False
,transformers,"[('num', ...), ('cat', ...)]"
,remainder,'drop'
,sparse_threshold,0.3
,n_jobs,None
,transformer_weights,None
,verbose,False
,verbose_feature_names_out,True


# MAKE PREDICTION ON TESTING

In [24]:
y_predtest=model_pipeline.predict(X_test)

# CHECK THE ACCURAY SCORE BETWEEN MODEL PREDICTON AND Y_TEST TO CHECK HOW THE  MODEL PEFORM ON UNSEEN DATA

In [25]:
accuracy_score(y_predtest,y_test)

0.8311688311688312

In [26]:
confusion_matrix(y_predtest,y_test)

array([[11,  0],
       [13, 53]])

In [27]:
 precision_score(y_predtest,y_test)

1.0

In [28]:
f1_score(y_predtest,y_test)

0.8907563025210085

In [177]:
!pip install imblearn

Defaulting to user installation because normal site-packages is not writeable


In [178]:
from imblearn.over_sampling import SMOTE

In [179]:
df=pd.read_csv("mEzNPN.csv")

In [180]:
df.Loan_Status=df.Loan_Status.replace({
    'Y':1,
    'N':0
})

C:\Users\PMLS\AppData\Local\Temp\ipykernel_3704\324800776.py:1: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df.Loan_Status=df.Loan_Status.replace({


In [181]:
X=df.drop('Loan_Status',axis=1)

In [182]:
y=df['Loan_Status']